In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "model" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing model/__init__.py")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import model.core as core
from model.core import STDPModel, STDPParams
from model.notebook import ModelWorkflow, WorkflowConfig
from model.fit import FitConfig
from model.data import load_standard_errors
from model.plotting_basic import configure_plot_style, evaluate_model, plot_fraction_comparison
from model.plotting_advanced import (
    composition_vs_tau,
    composition_vs_concentration,
    composition_vs_age,
    grid_survivor_composition,
    make_cmyk_rgb,
    show_rgb_heatmap,
    show_heatmap_with_keys_labeled,
    grid_descendant_composition,
    show_contour,
    J_discounted,
)
from model.objective import NUM_BOUNDS, _pack_params, _unpack_params
from model.uncertainty import (
    build_objectives,
    hessian_central,
    robust_inverse,
    compute_prediction_ci,
    profile_likelihood,
)

configure_plot_style("nature")

# ── Standard panel sizing for Inkscape panel arrangement ──
PANEL = 1.5  # base panel size in inches
MARGINS = dict(left=0.20, bottom=0.22, right=0.97, top=0.92)

PLOT_DIR = REPO_ROOT / "output" / "model-plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = REPO_ROOT / "model" / "fractions.csv"
PLOT_DIR

In [ ]:
MAIN_FIT_CONFIG = FitConfig(profile="custom", n_starts=240, maxiter=1500, seed=2025)


# Set True to ignore cached fit and re-run optimisation from scratch.
FORCE_REFIT = False

LOG_HAZARD_GLOBAL_DEFAULT = False
LOG_HAZARD_hS = False
LOG_HAZARD_hT = False
LOG_HAZARD_rST = False

core.set_log_hazard_flags(
    global_default=LOG_HAZARD_GLOBAL_DEFAULT,
    hS=LOG_HAZARD_hS,
    hT=LOG_HAZARD_hT,
    rST=LOG_HAZARD_rST,
)

HILL_HAZARD_GLOBAL_DEFAULT = True
HILL_HAZARD_hS = True
HILL_HAZARD_hT = True
HILL_HAZARD_rST = True

HILL_LAMBDA_CONSTRAINT = 1e6

core.set_hill_hazard_flags(
    global_default=HILL_HAZARD_GLOBAL_DEFAULT,
    hS=HILL_HAZARD_hS,
    hT=HILL_HAZARD_hT,
    rST=HILL_HAZARD_rST,
)

FREE_KEYS = [
    "w_lag", "lam_slow",
    "mu0", "mu24p",
    "kS_kT_ratio", "kT", "kST",
    "K", "KST",
    "n", "nST",
    "a50", "r0",
]
print("FREE_KEYS:", FREE_KEYS)

PARAMS0 = STDPParams(
    w_lag=0.95,
    lam_slow=0.1,
    k_lag=6,
    mu0=0.69,
    mu24p=0.21,
    kT=20.0,
    kS_kT_ratio=3.0,
    kST=1.91,
    K=15.0,
    KST=50.0,
    n=1.0,
    nST=1.0,
    a50=32.1,
    KS=None,   # fall back to shared K
    KT=None,   # fall back to shared K
)

K_LAG_VALUES = range(2, 15)

N_DRAWS = 5000
PROFILE_N_POINTS = 40
PROFILE_DELTA_LOG = 1.5

C_GRID = np.geomspace(1.0, 1000.0, 30)
TAU_GRID = np.linspace(0.0, 8.0, 30)
A_FIXED = 24.0

FIT_CACHE = PLOT_DIR / "fit_cache_hybrid.pkl"

In [ ]:
import pickle

workflow = ModelWorkflow(
    params=PARAMS0,
    config=WorkflowConfig(
        fit_config=MAIN_FIT_CONFIG,
        kappa=5000.0,
        lam_pen=1e4,
        lam_hill_constraint=HILL_LAMBDA_CONSTRAINT,
        lam_regime=1e4,
        rho=0.8,
        ages_for_pen=(24.0, 48.0, 72.0),
        class_weights=(1, 1, 1, 1),
    )
)

DATA = workflow.load_data(DATA_PATH)
FORCE_REFIT=True
if not FORCE_REFIT and FIT_CACHE.exists():
    with open(FIT_CACHE, "rb") as f:
        cached = pickle.load(f)
    best_params = cached["best_params"]
    best_x = cached["best_x"]
    free_keys = cached["free_keys"]
    best_value = cached["best_value"]
    workflow.model = STDPModel(best_params)
    model_fit = workflow.model
    print(f"Loaded cached fit from {FIT_CACHE.name}")
    print(f"  objective: {best_value:.6g}")
else:
    fit_result = workflow.fit(config=MAIN_FIT_CONFIG, free_keys=FREE_KEYS)
    best_params = fit_result.best_params
    best_x = fit_result.best_x
    free_keys = fit_result.free_keys
    best_value = fit_result.best_value
    model_fit = workflow.model
    with open(FIT_CACHE, "wb") as f:
        pickle.dump({
            "best_params": best_params,
            "best_x": best_x,
            "free_keys": free_keys,
            "best_value": best_value,
        }, f)
    print(f"Fit complete, cached to {FIT_CACHE.name}")
    print(f"  objective: {best_value:.6g}")

print(f"Loaded conditions: {len(DATA)}")

bound_rows = []
for key in free_keys:
    val = getattr(best_params, key)
    lo, hi = NUM_BOUNDS[key]
    in_bounds = (lo <= val <= hi)
    at_lower = abs(val - lo) <= 1e-12 * max(1.0, abs(lo))
    at_upper = abs(val - hi) <= 1e-12 * max(1.0, abs(hi))
    bound_rows.append({
        "param": key,
        "value": val,
        "lower": lo,
        "upper": hi,
        "in_bounds": in_bounds,
        "at_lower_bound": at_lower,
        "at_upper_bound": at_upper,
    })

bounds_df = pd.DataFrame(bound_rows).set_index("param")

display(pd.DataFrame([best_params.__dict__]).T.rename(columns={0: "value"}))
bounds_df

In [ ]:
print((pd.DataFrame([best_params.__dict__]).T.rename(columns={0: "value"}))
)

In [ ]:
# Build objective pieces (matching old script's uncertainty workflow)
objective_total, nll_only, pen_only, unpack = build_objectives(
    template=best_params,
    data=DATA,
    free_keys=free_keys,
    kappa=5000.0,
    lam_pen=1e4,
    lam_hill_constraint=HILL_LAMBDA_CONSTRAINT,
    lam_regime=1e4,
    rho=0.8,
    ages_for_pen=(24.0, 48.0, 72.0),
    class_weights=(1, 1, 1, 1),
)

x_hat = _pack_params(best_params, free_keys)
H_data = hessian_central(nll_only, x_hat)
H_total = hessian_central(objective_total, x_hat)

Sigma_data = robust_inverse(H_data)
Sigma_total = robust_inverse(H_total)

se_log = np.sqrt(np.diag(Sigma_data))
means_log = x_hat.copy()

ci95_lower = np.exp(means_log - 1.96 * se_log)
ci95_upper = np.exp(means_log + 1.96 * se_log)
mle_vals = np.exp(means_log)

summary_df = pd.DataFrame(
    {
        "param": free_keys,
        "mle": mle_vals,
        "se_log": se_log,
        "ci95_lo": ci95_lower,
        "ci95_hi": ci95_upper,
    }
).set_index("param")

curvature = np.diag(H_data)
ranking = sorted(zip(free_keys, curvature), key=lambda t: -t[1])

rng = np.random.default_rng(1)
draws_log = rng.multivariate_normal(mean=means_log, cov=Sigma_data, size=N_DRAWS)
draws = np.exp(np.clip(draws_log, -50, 50))

ci_df = compute_prediction_ci(draws_log, best_params, free_keys, DATA)

SE_DF = load_standard_errors(DATA_PATH)
df_fit = evaluate_model(model_fit, DATA, se_df=SE_DF)
df_fit = df_fit.merge(ci_df, on=["condition", "C", "tau", "age"], how="left")

In [ ]:
df_fit = evaluate_model(model_fit, DATA, se_df=SE_DF)
df_fit = df_fit.merge(ci_df, on=["condition", "C", "tau", "age"], how="left")

In [ ]:
df_fit.to_csv(PLOT_DIR / "model_fit_with_ci.csv", index=False)

In [ ]:
frac_specs = [
    ("incomplete", dict(use_log=True, title="Exposure Limited Survivor", xlim=(1e-4, 1.0), ylim=(3*1e-5, 1.0), n_sigma=2.96)),
    ("induced", dict(use_log=True, title="Transient Tolerant Survivor", xlim=(5e-3, 0.3), ylim=(5e-3, 0.3), n_sigma=2.96)),
    ("preexisting", dict(use_log=True, title="Persister", show_legend=False, xlim=(1e-5, 1.0), ylim=(1e-5, 1.0), n_sigma=2.96)),
    ("dead", dict(use_log=False, title="Dead (Inferred)", xlim=(2e-1, 1.0), ylim=(2e-1, 1.0), n_sigma=2.96)),
]


for frac, kwargs in frac_specs:
    out = PLOT_DIR / f"stdp_fit_{frac}.svg"
    fig, ax = plt.subplots(figsize=(PANEL, PANEL))
    fig.subplots_adjust(**MARGINS)
    fig, ax = plot_fraction_comparison(frac=frac, df_fit=df_fit, save_path=str(out), ax=ax, **kwargs)
    plt.show()
    print(f"Saved {out}")

In [ ]:
fig, ax = composition_vs_tau(model_fit, C=50.0, a=24.0, tau_grid=np.linspace(0, 8, 201), figsize=(PANEL*1., PANEL))
fig.subplots_adjust(**MARGINS)
fig.savefig(PLOT_DIR / "composition_tau.svg")
plt.show()

fig, ax = composition_vs_concentration(model_fit, tau=4.0, a=24.0, C_grid=np.geomspace(0.05, 1000, 241), figsize=(PANEL*1., PANEL))
fig.subplots_adjust(**MARGINS)
fig.savefig(PLOT_DIR / "composition_vs_C.svg")
plt.show()

fig, ax = composition_vs_age(model_fit, C=50.0, tau=4.0, a_grid=np.linspace(0.0, 96.0, 241), figsize=(PANEL*1., PANEL))
fig.subplots_adjust(**MARGINS)
fig.savefig(PLOT_DIR / "composition_vs_age.svg")
plt.show()

In [ ]:
# Composition stack plots — survivors only (no dead), rescaled to sum to 1
def _survivor_stack(model, x_grid, x_param, fixed, figsize, xlabel):
    """Stack incomplete/induced/preexisting normalised to survivor total."""
    inc, ind, pre = [], [], []
    for v in x_grid:
        kw = {x_param: v, **fixed}
        pr = model.predict_condition(**kw)
        s, t, d = float(pr.incomplete), float(pr.induced), float(pr.preexisting)
        total = s + t + d
        if total > 0:
            inc.append(s / total)
            ind.append(t / total)
            pre.append(d / total)
        else:
            inc.append(1 / 3)
            ind.append(1 / 3)
            pre.append(1 / 3)
    fig, ax = plt.subplots(figsize=figsize)
    ax.stackplot(
        x_grid,
        np.asarray(inc), np.asarray(ind), np.asarray(pre),
        labels=["Exposure limited", "Transient tolerant", "Persister"],
        colors=["#ff8ef475", "#ffff7c75", "#7cffff75"],
        alpha=0.85,
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Fraction of survivors")
    ax.set_xlim(x_grid.min(), x_grid.max())
    ax.set_ylim(0, 1)
    return fig, ax

tau_grid = np.linspace(0, 8, 201)
fig, ax = _survivor_stack(model_fit, tau_grid, "tau", dict(C=50.0, a=24.0), (PANEL, PANEL), "Treatment duration: τ (h)")
fig.subplots_adjust(**MARGINS)
fig.savefig(PLOT_DIR / "composition_tau_no_dead.svg")
plt.show()

C_grid = np.geomspace(0.05, 1000, 241)
fig, ax = _survivor_stack(model_fit, C_grid, "C", dict(tau=4.0, a=24.0), (PANEL, PANEL), "Antibiotic conc: C (μg/mL)")
ax.set_xscale("log")
fig.subplots_adjust(**MARGINS)
fig.savefig(PLOT_DIR / "composition_vs_C_no_dead.svg")
plt.show()

a_grid = np.linspace(0.0, 96.0, 241)
fig, ax = _survivor_stack(model_fit, a_grid, "a", dict(C=50.0, tau=4.0), (PANEL, PANEL), "Culture age: a (h)")
fig.subplots_adjust(**MARGINS)
fig.savefig(PLOT_DIR / "composition_vs_age_no_dead.svg")
plt.show()

In [ ]:
phi_S, phi_T, phi_D, surv = grid_survivor_composition(model_fit, C_GRID, TAU_GRID, A_FIXED)

# Composition only heatmap
rgb_comp = make_cmyk_rgb(phi_S, phi_T, phi_D, surv=None, mode="composition", gamma=1.0)
fig, ax = plt.subplots(figsize=(PANEL, PANEL))
fig.subplots_adjust(**MARGINS)
show_rgb_heatmap(C_GRID, TAU_GRID, rgb_comp, ax=ax, logC=True, title="CMYK composition: C=Dormant, M=Susceptible, Y=Tolerant")
plt.show()

# Survivor weighted heatmap
rgb_sw = make_cmyk_rgb(
    phi_S,
    phi_T,
    phi_D,
    surv=surv,
    mode="survivor_weighted",
    use_K=True,
    k_gamma=0.25,
    sat=1.0,
    gamma=2.0,
)
fig, ax = plt.subplots(figsize=(PANEL, PANEL))
fig.subplots_adjust(**MARGINS)
show_rgb_heatmap(C_GRID, TAU_GRID, rgb_sw, ax=ax, logC=True, title="CMYK × survivors (K optional): C=D, M=S, Y=T")
plt.show()

# Saved composition heatmap (as in original)
fig, ax = plt.subplots(figsize=(PANEL, PANEL))
fig.subplots_adjust(**MARGINS)
show_rgb_heatmap(C_GRID, TAU_GRID, rgb_comp, ax=ax, logC=True, title="Fractional survivor composition")
fig.savefig(PLOT_DIR / "heatmap_no_keys.svg")
plt.show()

# Line plot vs C
fig, ax = plt.subplots(figsize=(PANEL*0.75, PANEL*0.5*0.75))
fig.subplots_adjust(**MARGINS)
ax.plot(C_GRID, phi_S[14, :], label="Susceptible (S)", c="#C900C9")
ax.plot(C_GRID, phi_T[14, :], label="Tolerant (T)", c="#F7BB25")
ax.plot(C_GRID, phi_D[14, :], label="Dormant (D)", c="#00CBCB")
ax.set_xscale("log")
ax.set_xlabel("C (µg/mL)")
ax.set_ylabel("Survivor fraction")
ax.set_yticks([0, 0.5, 1])
ax.set_xticks([1, 10, 100, 1000])
fig.savefig(PLOT_DIR / "composition_vs_C_line.svg")
plt.show()

# Line plot vs tau
fig, ax = plt.subplots(figsize=(PANEL*0.75, PANEL*0.5*0.75))
fig.subplots_adjust(**MARGINS)
ax.plot(TAU_GRID, phi_S[:, 14].T, label="Susceptible (S)", c="#C900C9")
ax.plot(TAU_GRID, phi_T[:, 14].T, label="Tolerant (T)", c="#F7BB25")
ax.plot(TAU_GRID, phi_D[:, 14].T, label="Dormant (D)", c="#00CBCB")
ax.set_xlabel("τ (h)")
ax.set_ylabel("Survivor fraction")
ax.set_xticks([0, 4, 8])
ax.set_yticks([0, 0.5, 1])
fig.savefig(PLOT_DIR / "composition_vs_tau_line.svg")
plt.show()

# Survival contour
fig, ax = show_contour(
    C_GRID,
    TAU_GRID,
    surv,
    logC=True,
    levels=[0, 0.001, 0.01, 0.1, 0.5, 0.9, 1],
    colour_gamma=0.2,
    figsize=(PANEL, PANEL),
)
fig.subplots_adjust(**MARGINS)
ax.set_title("Survival rate isoclines")
fig.savefig(PLOT_DIR / "contour_survival.svg")
plt.show()

In [ ]:
u_fixed = 1e2
g_rate = 1.0

phiS_u, phiT_u, phiD_u, R_u = grid_descendant_composition(
    model_fit,
    C_GRID,
    TAU_GRID,
    A_FIXED,
    u_fixed,
    g_rate,
)

# Composition-only descendants heatmap
rgb_post_comp = make_cmyk_rgb(
    phiS_u,
    phiT_u,
    phiD_u,
    surv=None,
    mode="composition",
    gamma=1.0,
    use_K=True,
    k_gamma=1.0,
    sat=1.15,
)
fig, _ = show_heatmap_with_keys_labeled(
    C_GRID,
    TAU_GRID,
    rgb_post_comp,
    k_gamma=1.0,
    title_heatmap=f"Descendant composition at u={u_fixed:.0f} h (CMY: D,S,T)",
    bar_label="brightness unused",
    figsize=(3 * PANEL, PANEL),
    gridspec_kw=dict(left=0.07, right=0.97, bottom=0.22, top=0.92),
)
fig.savefig(PLOT_DIR / f"post_treatment_composition_u{int(u_fixed)}h.svg")
plt.show()

# Weighted descendants heatmap
R_norm = R_u / max(np.max(R_u), 1e-12)
k_gamma_ptd = 0.35
rgb_post_weighted = make_cmyk_rgb(
    phiS_u,
    phiT_u,
    phiD_u,
    surv=np.clip(R_norm, 0.0, 1.0),
    mode="survivor_weighted",
    use_K=True,
    k_gamma=k_gamma_ptd,
    gamma=2.0,
    sat=1.15,
)
fig, _ = show_heatmap_with_keys_labeled(
    C_GRID,
    TAU_GRID,
    rgb_post_weighted,
    k_gamma=k_gamma_ptd,
    title_heatmap=f"Descendant composition × regrown mass at u={u_fixed:.0f} h",
    bar_label=r"brightness ∝ $N(u)/N_0$ (grid-normalized)",
    figsize=(3 * PANEL, PANEL),
    gridspec_kw=dict(left=0.07, right=0.97, bottom=0.22, top=0.92),
)
fig.savefig(PLOT_DIR / f"post_treatment_weighted_u{int(u_fixed)}h.svg")
plt.show()

# Descendant T-fraction contour
fig, ax = show_contour(
    C_GRID,
    TAU_GRID,
    phiT_u,
    logC=True,
    levels=[0, 0.001, 0.01, 0.1, 0.5, 0.9, 0.99, 1],
    colour_gamma=0.6,
    figsize=(PANEL, PANEL),
)
fig.subplots_adjust(**MARGINS)
ax.set_title("Fraction of survivors descended from T")
fig.savefig(PLOT_DIR / "contour_fraction_T_descendants.svg")
plt.show()

In [ ]:

from matplotlib.colors import PowerNorm

# Compute descendant shares at u≈0 (use small epsilon to avoid 0/0 at τ=0)
phiS_0, phiT_0, phiD_0, _ = grid_descendant_composition(
    model_fit, C_GRID, TAU_GRID, A_FIXED, u=1e-6, g=g_rate,
)

shared_levels = [0, 0.01, 0.1, 0.5, 0.9, 0.99, 1.0]
x_log = np.log10(C_GRID)
extent = [x_log.min(), x_log.max(), TAU_GRID.min(), TAU_GRID.max()]

fig, axes = plt.subplots(1, 3, figsize=(3 * PANEL + 0.6, PANEL + 0.15),
                         gridspec_kw=dict(width_ratios=[1, 1, 0.08]))
fig.subplots_adjust(left=0.09, bottom=0.22, right=0.88, top=0.85, wspace=0.35)

panels = [
    (phiT_0, r"$w_T$ at drug removal ($u = 0$)"),
    (phiT_u, f"$w_T$ after regrowth ($u = {u_fixed:.0f}$ h)"),
]

for ax_i, (data, title) in enumerate(panels):
    ax = axes[ax_i]
    norm = PowerNorm(gamma=0.6, vmin=0.0, vmax=1.0)
    cf = ax.contourf(data, origin="lower", extent=extent, levels=shared_levels,
                     cmap="coolwarm", norm=norm)
    cnt = ax.contour(data, origin="lower", colors="k", extent=extent,
                     levels=shared_levels, norm=norm, linewidths=0.5)

    # Label contours at interior midpoints
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    mx = 0.15 * (xlim[1] - xlim[0])
    my = 0.15 * (ylim[1] - ylim[0])
    manual_pts = []
    for segs in cnt.allsegs:
        for seg in segs:
            inside = ((seg[:, 0] >= xlim[0] + mx) & (seg[:, 0] <= xlim[1] - mx)
                      & (seg[:, 1] >= ylim[0] + my) & (seg[:, 1] <= ylim[1] - my))
            if inside.any():
                idx = np.where(inside)[0]
                manual_pts.append((seg[idx[len(idx) // 2], 0], seg[idx[len(idx) // 2], 1]))
                break
    if manual_pts:
        ax.clabel(cnt, cnt.levels, inline=True, fontsize=5, manual=manual_pts)

    # Axis formatting
    exp_lo = int(np.floor(np.log10(C_GRID.min())))
    exp_hi = int(np.ceil(np.log10(C_GRID.max())))
    ticks = [10**k for k in range(exp_lo, exp_hi + 1)]
    ax.set_xticks(np.log10(ticks))
    ax.set_xticklabels([f"{t:g}" for t in ticks])
    ax.set_xlabel("C (µg/mL)")
    ax.set_yticks(np.arange(0, 9, 2))
    ax.set_title(title, fontsize=6)
    if ax_i == 0:
        ax.set_ylabel("τ (h)")
    else:
        ax.set_yticklabels([])

# Shared colorbar
cbar = fig.colorbar(cf, cax=axes[2])
cbar.set_label("Fraction T", fontsize=6)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

fig.savefig(PLOT_DIR / "T_fraction_u0_vs_regrowth.svg")
plt.show()

In [ ]:

from scipy.optimize import brentq
import matplotlib.patches as mpatches
from model.plotting_advanced import _regularized_gamma_P

g_cross = 1.0  # h⁻¹
w_lag = model_fit.p.w_lag
k_lag = model_fit.p.k_lag
lam_erl = model_fit.erlang_rate(A_FIXED)
lam_slow = model_fit.p.lam_slow

def _Jg_inf(tau):
    """Asymptotic J_g(∞) for hybrid Erlang + Exponential lag."""
    # Erlang: w * (lam/(lam+g))^k * exp(g*tau) * [1 - P(k,(lam+g)*tau)]
    alpha_e = lam_erl + g_cross
    ratio = lam_erl / alpha_e
    J_erl = w_lag * (ratio ** k_lag) * np.exp(g_cross * tau) * (
        1.0 - _regularized_gamma_P(k_lag, alpha_e * tau)
    )
    # Exponential: (1-w) * lam_s * exp(-lam_s*tau) / (g + lam_s)
    J_exp = (1.0 - w_lag) * lam_slow * np.exp(-lam_slow * tau) / (g_cross + lam_slow)
    return J_erl + J_exp

t_cross = np.full((len(TAU_GRID), len(C_GRID)), np.nan)

for i, tau in enumerate(TAU_GRID):
    if tau < 1e-9:
        continue
    for j, C in enumerate(C_GRID):
        pr = model_fit.predict_condition(C=C, tau=tau, a=A_FIXED)
        pi_T = float(pr.induced)

        if pi_T <= 1e-15:
            t_cross[i, j] = 0.0
            continue

        jinf = _Jg_inf(tau)
        if jinf <= pi_T:
            t_cross[i, j] = np.inf
            continue

        f = lambda u: J_discounted(u, tau, w_lag, k_lag, lam_erl, lam_slow, g_cross) - pi_T
        u_hi = 10.0
        for _ in range(30):
            if J_discounted(u_hi, tau, w_lag, k_lag, lam_erl, lam_slow, g_cross) >= pi_T:
                break
            u_hi *= 2
        else:
            t_cross[i, j] = np.inf
            continue
        t_cross[i, j] = brentq(f, 0.0, u_hi, xtol=1e-6)

never_mask = np.isinf(t_cross)

# Replace inf with NaN so contour ignores those; cap for display
NEVER_SENTINEL = 500.0
plot_data = t_cross.copy()
plot_data[never_mask] = NEVER_SENTINEL
plot_data[np.isnan(plot_data)] = NEVER_SENTINEL

fig, ax = show_contour(
    C_GRID, TAU_GRID, plot_data,
    logC=True,
    levels=[0, 0.1, 0.5, 1, 2, 5, 10, 50],
    colour_gamma=0.5,
    figsize=(PANEL, PANEL),
)
fig.subplots_adjust(**MARGINS)

# Overlay "never overtakes" boundary
if never_mask.any():
    x_log = np.log10(C_GRID)
    extent = [x_log.min(), x_log.max(), TAU_GRID.min(), TAU_GRID.max()]
    ax.contourf(never_mask.astype(float), origin="lower", extent=extent,
                levels=[0.5, 1.5], colors=["white"], alpha=0.85)
    ax.contour(never_mask.astype(float), origin="lower", extent=extent,
               levels=[0.5], colors=["k"], linewidths=0.8, linestyles="--")

ax.set_title(f"Time for D to overtake T (g = {g_cross} h$^{{-1}}$)", fontsize=6)

fig.savefig(PLOT_DIR / "overtake_time_D_vs_T.svg")
plt.show()

n_never = np.sum(never_mask)
n_valid = np.sum(~np.isnan(t_cross))
finite_mask = np.isfinite(t_cross) & (t_cross > 0)
print(f"T dominates indefinitely: {n_never}/{n_valid} grid points"
      f" ({100*n_never/n_valid:.1f}%)")
if finite_mask.any():
    print(f"Overtake time range: {t_cross[finite_mask].min():.2f}"
          f" – {t_cross[finite_mask].max():.2f} h")

In [ ]:


from scipy.optimize import brentq
from model.plotting_advanced import _regularized_gamma_P

g_cross = 1.0  # h⁻¹, post-treatment growth rate

w_lag    = model_fit.p.w_lag
k_lag    = model_fit.p.k_lag
lam_erl  = model_fit.erlang_rate(A_FIXED)
lam_slow = model_fit.p.lam_slow

C_fine   = np.geomspace(1.0, 1000.0, 60)
tau_fine = np.linspace(0.1, 8.0, 60)


t_T_dominance = np.full((len(tau_fine), len(C_fine)), np.nan)

# Also store the survivor composition for the boundary overlay
piD_grid = np.full_like(t_T_dominance, np.nan)
piT_grid = np.full_like(t_T_dominance, np.nan)

for i, tau in enumerate(tau_fine):
    for j, C in enumerate(C_fine):
        pr = model_fit.predict_condition(C=C, tau=tau, a=A_FIXED)
        pi_T = float(pr.induced)
        pi_D = float(pr.preexisting)
        piD_grid[i, j] = pi_D
        piT_grid[i, j] = pi_T

        # Only the "blue" region: D > T at treatment end
        if pi_D <= pi_T or pi_T < 1e-15:
            continue

        # Asymptotic J_g(∞): total growth-discounted D contribution
        alpha_e = lam_erl + g_cross
        ratio = lam_erl / alpha_e
        J_erl_inf = w_lag * (ratio ** k_lag) * np.exp(g_cross * tau) * (
            1.0 - _regularized_gamma_P(k_lag, alpha_e * tau)
        )
        J_exp_inf = (1.0 - w_lag) * lam_slow * np.exp(-lam_slow * tau) / (g_cross + lam_slow)
        Jg_inf = J_erl_inf + J_exp_inf

        if Jg_inf <= pi_T:
            # D never catches up → T dominates forever
            t_T_dominance[i, j] = np.inf
            continue

        f = lambda u: J_discounted(u, tau, w_lag, k_lag, lam_erl, lam_slow, g_cross) - pi_T
        u_hi = 10.0
        for _ in range(30):
            if J_discounted(u_hi, tau, w_lag, k_lag, lam_erl, lam_slow, g_cross) >= pi_T:
                break
            u_hi *= 2
        else:
            t_T_dominance[i, j] = np.inf
            continue
        try:
            t_T_dominance[i, j] = brentq(f, 0.0, u_hi, xtol=1e-6)
        except ValueError:
            t_T_dominance[i, j] = np.inf

from matplotlib.colors import PowerNorm

valid  = ~np.isnan(t_T_dominance)
never  = np.isinf(t_T_dominance)

plot_data = t_T_dominance.copy()
plot_data[never] = np.nan
plot_data[~valid] = np.nan

x_log = np.log10(C_fine)
extent = [x_log.min(), x_log.max(), tau_fine.min(), tau_fine.max()]

fig, ax = plt.subplots(figsize=(PANEL + 0.3, PANEL))
fig.subplots_adjust(left=0.22, bottom=0.22, right=0.82, top=0.88)

levels = [0, 0.5, 1, 2, 5, 10, 20, 50, 100]
norm = PowerNorm(gamma=0.5, vmin=0, vmax=100)

cf = ax.contourf(plot_data, origin="lower", extent=extent,
                 levels=levels, cmap="viridis_r", norm=norm, extend="max")
ct = ax.contour(plot_data, origin="lower", extent=extent,
                levels=levels, colors="k", linewidths=0.4, norm=norm)
ax.clabel(ct, ct.levels[1:], inline=True, fontsize=4, fmt="%.0f")

# "T dominates forever" region (permanent blue → yellow)
if never.any():
    ax.contourf(never.astype(float), origin="lower", extent=extent,
                levels=[0.5, 1.5], colors=["#ffffaa"], alpha=0.6)
    ax.contour(never.astype(float), origin="lower", extent=extent,
               levels=[0.5], colors=["goldenrod"], linewidths=0.8, linestyles="--")

# Grey out region where T ≥ D already at treatment end (not "blue")
outside = (~valid).astype(float)
ax.contourf(outside, origin="lower", extent=extent,
            levels=[0.5, 1.5], colors=["#e0e0e0"], alpha=0.5)

# π_D = π_T boundary (edge of "blue" region on CMYK)
D_dom = (piD_grid > piT_grid).astype(float)
ax.contour(D_dom, origin="lower", extent=extent,
           levels=[0.5], colors=["k"], linewidths=1.0, linestyles="-")

# Axis formatting
exp_lo = int(np.floor(np.log10(C_fine.min())))
exp_hi = int(np.ceil(np.log10(C_fine.max())))
ticks = [10**k for k in range(exp_lo, exp_hi + 1)]
ax.set_xticks(np.log10(ticks))
ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.set_xlabel("C (µg/mL)")
ax.set_ylabel("τ (h)")
ax.set_title(f"Duration of T-dominance in regrowth\n(D-dominant region, a = {A_FIXED:.0f} h, g = {g_cross} h⁻¹)",
             fontsize=5)

cbar = fig.colorbar(cf, ax=ax, pad=0.02)
cbar.set_label("Time until D reclaims dominance (h)", fontsize=5)

fig.savefig(PLOT_DIR / "contour_T_dominance_duration.svg")
plt.show()

# Summary
finite_mask = np.isfinite(t_T_dominance) & ~np.isnan(t_T_dominance)
print(f"D-dominant region (blue on CMYK): {valid.sum()} / {valid.size} grid points")
print(f"  T dominates forever (blue→yellow): {never.sum()} ({100*never.sum()/max(valid.sum(),1):.1f}%)")
if finite_mask.any():
    print(f"  D reclaims dominance after: {t_T_dominance[finite_mask].min():.2f}"
          f" – {t_T_dominance[finite_mask].max():.2f} h")

In [ ]:
from matplotlib.colors import PowerNorm
import matplotlib.patches as mpatches
import matplotlib
fig, ax = plt.subplots(figsize=(PANEL + 0.3, PANEL))
fig.subplots_adjust(left=0.22, bottom=0.22, right=0.82, top=0.88)

x_log = np.log10(C_fine)
extent = [x_log.min(), x_log.max(), tau_fine.min(), tau_fine.max()]

hm_data = t_T_dominance.copy()
never   = np.isinf(t_T_dominance)
outside = np.isnan(t_T_dominance)
hm_data[never | outside] = np.nan

# Background: grey = not blue region, yellow = T dominates forever
bg = np.full_like(t_T_dominance, np.nan)
bg[outside] = 0.0   # grey
bg[never]   = 1.0   # yellow (permanent blue→yellow)

cmap_bg = matplotlib.colors.ListedColormap(["#e0e0e0", "#ffffaa"])
ax.imshow(bg, origin="lower", extent=extent, aspect="auto",
          cmap=cmap_bg, vmin=0, vmax=1, interpolation="nearest")

# Main heatmap: time until D reclaims dominance
norm = PowerNorm(gamma=0.5, vmin=0.01, vmax=100)
im = ax.imshow(hm_data, origin="lower", extent=extent, aspect="auto",
               cmap="viridis_r", norm=norm, interpolation="bilinear")

# Axis formatting
exp_lo = int(np.floor(np.log10(C_fine.min())))
exp_hi = int(np.ceil(np.log10(C_fine.max())))
ticks = [10**k for k in range(exp_lo, exp_hi + 1)]
ax.set_xticks(np.log10(ticks))
ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.set_xlabel("C (µg/mL)")
ax.set_ylabel("τ (h)")
ax.set_title(f"Duration of T-dominance in regrowth\n(D-dominant region, a = {A_FIXED:.0f} h, g = {g_cross} h⁻¹)",
             fontsize=5)

cbar = fig.colorbar(im, ax=ax, pad=0.02)
cbar.set_label("Time until D reclaims dominance (h)", fontsize=5)

patches = [
    mpatches.Patch(facecolor="#ffffaa", edgecolor="k", linewidth=0.5,
                   label="T dominates forever"),
    mpatches.Patch(facecolor="#e0e0e0", edgecolor="k", linewidth=0.5,
                   label="T ≥ D at treatment end"),
]
ax.legend(handles=patches, fontsize=4, loc="upper left", framealpha=0.8)

fig.savefig(PLOT_DIR / "heatmap_T_dominance_duration.svg")
plt.show()

In [ ]:

from matplotlib.colors import LogNorm

g_sens = 1.0  # h⁻¹
w_lag_s = model_fit.p.w_lag
k_lag_s = model_fit.p.k_lag
lam_erl_s = model_fit.erlang_rate(A_FIXED)
lam_slow_s = model_fit.p.lam_slow

def _Jg_inf(tau):
    """Asymptotic J_g(∞) for hybrid Erlang + Exponential lag."""
    alpha_e = lam_erl_s + g_sens
    ratio = lam_erl_s / alpha_e
    J_erl = w_lag_s * (ratio ** k_lag_s) * np.exp(g_sens * tau) * (
        1.0 - _regularized_gamma_P(k_lag_s, alpha_e * tau)
    )
    J_exp = (1.0 - w_lag_s) * lam_slow_s * np.exp(-lam_slow_s * tau) / (g_sens + lam_slow_s)
    return J_erl + J_exp

M_star = np.full((len(TAU_GRID), len(C_GRID)), np.nan)

for i, tau in enumerate(TAU_GRID):
    if tau < 1e-9:
        continue
    jinf = _Jg_inf(tau)
    for j, C in enumerate(C_GRID):
        pr = model_fit.predict_condition(C=C, tau=tau, a=A_FIXED)
        pi_T = float(pr.induced)

        if jinf < 1e-30:
            M_star[i, j] = np.inf if pi_T > 1e-15 else np.nan
        elif pi_T < 1e-15:
            M_star[i, j] = 0.0
        else:
            M_star[i, j] = pi_T / jinf

# ── Plot with log-scale norm ──────────────────────────────────────────────
CAP = 1e8
plot_data = np.clip(np.where(np.isfinite(M_star), M_star, CAP), 1e-2, CAP)

x_log = np.log10(C_GRID)
extent = [x_log.min(), x_log.max(), TAU_GRID.min(), TAU_GRID.max()]

levels = [0.1, 1, 10, 100, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8]
norm = LogNorm(vmin=0.1, vmax=CAP)

fig, ax = plt.subplots(figsize=(PANEL, PANEL))
fig.subplots_adjust(**MARGINS)

cf = ax.contourf(plot_data, origin="lower", extent=extent, levels=levels,
                 cmap="coolwarm", norm=norm)
cnt = ax.contour(plot_data, origin="lower", extent=extent, levels=levels,
                 colors="k", linewidths=0.4, norm=norm)

# Label the M*=1 boundary prominently
cnt1 = ax.contour(plot_data, origin="lower", extent=extent, levels=[1],
                  colors="k", linewidths=1.2, linestyles="-")
if cnt1.allsegs[0]:
    ax.clabel(cnt1, [1], inline=True, fontsize=6, fmt={1: "$M^*=1$"})

# Label other contours at interior midpoints
xlim, ylim = ax.get_xlim(), ax.get_ylim()
mx = 0.15 * (xlim[1] - xlim[0])
my = 0.15 * (ylim[1] - ylim[0])
manual_pts = []
for segs in cnt.allsegs:
    for seg in segs:
        inside = ((seg[:, 0] >= xlim[0] + mx) & (seg[:, 0] <= xlim[1] - mx)
                  & (seg[:, 1] >= ylim[0] + my) & (seg[:, 1] <= ylim[1] - my))
        if inside.any():
            idx = np.where(inside)[0]
            manual_pts.append((seg[idx[len(idx) // 2], 0], seg[idx[len(idx) // 2], 1]))
            break
if manual_pts:
    ax.clabel(cnt, cnt.levels, inline=True, fontsize=5, manual=manual_pts)

# Axis formatting
exp_lo = int(np.floor(np.log10(C_GRID.min())))
exp_hi = int(np.ceil(np.log10(C_GRID.max())))
ticks = [10**k for k in range(exp_lo, exp_hi + 1)]
ax.set_xticks(np.log10(ticks))
ax.set_xticklabels([f"{t:g}" for t in ticks])
ax.set_xlabel("C (µg/mL)")
ax.set_ylabel("τ (h)")
ax.set_yticks(np.arange(0, 9, 2))
ax.set_title(
    r"Fold increase in $\pi_D$ for D to overtake T" + "\n"
    + f"($g = {g_sens}$" + r" h$^{-1}$, $a = 24$ h)",
    fontsize=5,
)

fig.savefig(PLOT_DIR / "persister_multiplier_sensitivity.svg")
plt.show()

# Print key values
print("M* at selected conditions (a = 24 h):")
for C_q in [10, 25, 50, 100, 250]:
    j_c = np.argmin(np.abs(C_GRID - C_q))
    for tau_q in [2, 4, 6]:
        i_t = np.argmin(np.abs(TAU_GRID - tau_q))
        m = M_star[i_t, j_c]
        tag = " (D already dominates)" if m < 1 else ""
        print(f"  C={C_GRID[j_c]:.0f}, τ={TAU_GRID[i_t]:.1f}:  M* = {m:.1g}{tag}")

In [ ]:
# Recompute a local covariance around k-lag-selected best parameters for marginals.
objective_total_k, nll_only_k, _, _ = build_objectives(
    template=best_params,
    data=DATA,
    free_keys=free_keys,
    kappa=5000.0,
    lam_pen=1e4,
    lam_hill_constraint=HILL_LAMBDA_CONSTRAINT,
    lam_regime=1e4,
    rho=0.8,
    ages_for_pen=(24.0, 48.0, 72.0),
    class_weights=(1, 1, 1, 1),
)

x_hat_k = _pack_params(best_params, free_keys)
H_data_k = hessian_central(nll_only_k, x_hat_k)
Sigma_data_k = robust_inverse(H_data_k)

rng = np.random.default_rng(1)
draws_log_k = rng.multivariate_normal(mean=x_hat_k, cov=Sigma_data_k, size=N_DRAWS)
draws_k = np.exp(np.clip(draws_log_k, -50, 50))
mle_vals_k = np.exp(x_hat_k)

for j, name in enumerate(free_keys):
    if name == "r0":
        continue
    fig, ax = plt.subplots(figsize=(PANEL, PANEL))
    fig.subplots_adjust(**MARGINS)
    col_vals = draws_k[:, j]
    col_vals = col_vals[np.isfinite(col_vals)]
    ax.hist(col_vals, bins=40, density=True)
    ax.axvline(mle_vals_k[j], linestyle="--", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("value")
    ax.set_ylabel("density")
    fig.savefig(PLOT_DIR / f"param_marginal_{name}.svg")
    plt.show()

import seaborn as sns

plt.figure(figsize=(12, 12))
df_draws_k = pd.DataFrame(draws_k, columns=free_keys)
sns.pairplot(df_draws_k, corner=True, plot_kws={"alpha": 0.1 })
for i, name_i in enumerate(free_keys):
    for j, name_j in enumerate(free_keys):
        if j >= i:
            continue
        plt.gca().scatter(mle_vals_k[i], mle_vals_k[j], c="red", marker="X", s=100)
plt.suptitle("Pairwise scatter of parameter draws (red X = MLE)", y=1.02)
plt.tight_layout()